# Ensemble warrant gap + E/F distillation (Mezzanine pattern) on MD17

Runtime → Change runtime type → **GPU (T4)**. Upload `mezzanine_ensemble_ef_runner.py` in the next cell.

Defaults are the 1000-config MD17 benchmark. On an A100 use the recipe in cell 4 (`--batch 1024 --hidden 128 --parallel 8`): at the defaults the run is kernel-launch-bound and any GPU sits near idle.
Use `--quick` first (~1 min) to confirm everything runs.

In [ ]:
from google.colab import files
up = files.upload()            # select mezzanine_ensemble_ef_runner.py
assert 'mezzanine_ensemble_ef_runner.py' in up, 'upload the runner file'
import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())

In [ ]:
# Smoke test first (~2 min), then the real run.
!python mezzanine_ensemble_ef_runner.py --out runs/quick --molecule ethanol --quick 2>&1 | grep -v '] step '

In [ ]:
RUN = 'runs/ethanol_a100'
# A100: bigger batch/model so each kernel does real work, and the 8 members + 4 sweep fits run concurrently.
!python mezzanine_ensemble_ef_runner.py --out $RUN --molecule ethanol \
    --batch 1024 --hidden 128 --n_int 4 --steps 6000 --members 8 --parallel 8 \
    --n_unlabeled 8000 --n_test 2000 2>&1 | grep -v '] step '

# T4 / smaller GPU:  --batch 256 --hidden 96 --n_int 3 --steps 4000 --members 5 --parallel 4
# Variants worth running next:
#   --n_unlabeled 0                       pure-Mezzanine setting (no extra teacher-labelled configs)
#   --hard_label_weight 0.0               pure soft distillation
#   --molecule aspirin --steps 8000       harder molecule (21 atoms)
#   --npz path/to/rmd17_ethanol.npz       revised MD17 (cleaner labels)
#   --student_hidden 64 --student_int 2   compressed student
#   --tf32                                faster matmuls; equivariance criterion relaxes to 1e-2


In [ ]:
import json, pandas as pd
from IPython.display import Image, display
res = json.load(open(f'{RUN}/results.json'))
print('verdict:', res['make_break'])
print('warrant gap:', res['teacher']['warrant_gap'])
display(pd.read_csv(f'{RUN}/summary.csv'))
for f in ['fig_pareto', 'fig_parity', 'fig_gap', 'fig_md', 'fig_train']:
    display(Image(f'{RUN}/{f}.png'))

In [ ]:
# Everything needed for your own plots:
import numpy as np
P = np.load(f'{RUN}/predictions_test.npz')
print({k: P[k].shape for k in P.files})     # E_members [M,N], F_members [M,N,A,3], E_student, F_student, E_true, F_true ...

# Download the whole run directory
!zip -qr {RUN.replace('/', '_')}.zip $RUN && echo zipped
from google.colab import files; files.download(f"{RUN.replace('/', '_')}.zip")